**PART 1.  CONVOLUTION**

1.1  THE PROBLEM WITH FLATTENING  (you met it on Day 16)
-------------------------------------------------------------
On Day 16 each 8x8 digit became a flat list of 64 numbers.  That works for a
tiny image, but it has two problems:
  (a) It throws away the neighbourhood.  Pixel (3,4) and pixel (3,5) are
      neighbours, but a Linear layer sees them as two unrelated inputs.
  (b) The size explodes.  A 224 x 224 colour photo has 224 x 224 x 3 = 150,528
      numbers.  One hidden layer of 128 neurons would then have
      150,528 x 128 + 128 = 19,267,712 weights.  Far too many for 1,000 photos.

The CNN idea:  look at SMALL patches, and use the SAME small set of weights
everywhere on the image.  A tiny set of weights, slid across the picture,
detects "an edge here", "an edge there".  This is called WEIGHT SHARING.


1.2  THE SLIDING WINDOW (a convolution) BY HAND
-------------------------------------------------------------
  - An IMAGE is a table of numbers (brightness).
  - A KERNEL (also called a FILTER) is a small table of weights, here 3 x 3.
  - To get one output number: put the kernel on a 3 x 3 window of the image,
    multiply cell by cell, and ADD everything.
  - Slide the window one step to the right, repeat.  At the end of a row, go
    down one row.  The results form the OUTPUT table (the FEATURE MAP).

  (Strictly this is "cross-correlation"; every deep-learning library calls it a
  convolution and does not flip the kernel.  We follow the libraries.)

  Our 5 x 5 image (0 = dark, 3 = bright; a bright region on the right):

         c0  c1  c2  c3  c4
     r0   0   0   1   3   3
     r1   0   1   1   3   3
     r2   0   0   2   3   3
     r3   0   1   1   3   2
     r4   0   0   1   3   3

  Our kernel K (it computes "right column minus left column"):

         -1   0   1
         -1   0   1
         -1   0   1

  WORKED EXAMPLE, the output cell (0,0):  the window is rows 0-2, columns 0-2

         0 0 1          -1 0 1
         0 1 1    x     -1 0 1
         0 0 2          -1 0 1

  row 0:  0x(-1) + 0x0 + 1x1 = 1
  row 1:  0x(-1) + 1x0 + 1x1 = 1
  row 2:  0x(-1) + 0x0 + 2x1 = 2
  SUM = 4   ->  output(0,0) = 4

  EX 1 (paper, 10 min).  Compute the other 8 cells and fill the 3 x 3 output:

         output(0,0)=4    output(0,1)=8     output(0,2)=5
         output(1,0)=4    output(1,1)=7     output(1,2)=4
         output(2,0)=4    output(2,1)=8     output(2,2)=4

  Rule to write on your page:   the window for output(i, j) is
  image rows i..i+2 and columns j..j+2.


1.3  WHAT THE KERNEL DOES
-------------------------------------------------------------
K gives a BIG number where the right side of a window is brighter than the
left side: a vertical edge that goes from dark to bright.  Look at your output:
the numbers are large because our image has exactly that edge (left dark,
right bright).  A flat region (all the same value) gives 0.

A different kernel finds a different pattern.  The kernel H below finds a
horizontal edge ("top minus bottom"):

         1  1  1
         0  0  0
        -1 -1 -1

  EX 2 (paper, 5 min).  Apply H to the SAME image, but compute ONLY the centre
  output cell (1,1).  (Window = rows 1-3, columns 1-3.)
	- for (1, 1) = 0

  KEY IDEA:  in a CNN nobody types the kernel numbers.  They are the WEIGHTS.
  Gradient descent (Day 15) learns them.  In the first layer the network usually
  discovers edge detectors on its own; in deeper layers it combines edges into
  corners, then into parts, then into objects.

1.4  CHANNELS AND PARAMETER COUNT
-------------------------------------------------------------
  - A colour image has 3 channels (red, green, blue).  A kernel must have the
    same depth as its input: a 3 x 3 kernel on a 3-channel image has 3 x 3 x 3
    = 27 weights, and it still produces ONE output number per position.
  - One kernel gives ONE feature map.  A layer with 16 kernels gives 16 feature
    maps (16 output channels).
  - PARAMETERS of Conv2d(in_channels, out_channels, kernel_size):
        out_channels x in_channels x k x k   (weights)   +   out_channels  (biases)
    Example:  Conv2d(3, 16, 3)  ->  16 x 3 x 3 x 3 + 16 = 448.
    (In PyTorch the weight tensor has shape (16, 3, 3, 3).)
  - Compare: Linear(784, 128) alone has 784 x 128 + 128 = 100,480 parameters.

1.5  PADDING AND STRIDE
-------------------------------------------------------------
  - Without padding, the output is SMALLER than the input (5 x 5 became 3 x 3),
    and border pixels are used less often than centre pixels.
  - PADDING p: add p rings of zeros around the image.  With a 3 x 3 kernel,
    p = 1 keeps the size the same ("same" padding).  The formula for odd k is
    p = (k - 1) / 2.
  - STRIDE s: move the window s cells at a time instead of 1.  A stride of 2
    roughly halves the output size.

  THE OUTPUT-SIZE FORMULA  (write it on the first page of your notes):

        out = floor( (n + 2p - k) / s ) + 1

        n = input size,  k = kernel size,  p = padding,  s = stride

  Check with the example:  n=5, k=3, p=0, s=1  ->  floor(2/1) + 1 = 3.   OK.

  EX 3 (paper, 10 min).  Use the formula:
      - n=28, k=3, p=0, s=1  ->  26

      - n=28, k=3, p=1, s=1  ->  28

      - n=28, k=3, p=0, s=2  ->  13

      - n=32, k=5, p=2, s=1  ->  32

      - n=32, k=5, p=0, s=2  ->  14

      - n=64, k=7, p=3, s=2  ->  32

      - n=14, k=3, p=1, s=1  ->  14
      

  EX 4 (paper, 10 min).  Re-do the worked image with STRIDE 2 and no padding.
  Which output cells are kept?  (Hint: the windows start at columns 0 and 2 and
  rows 0 and 2.)  Write the resulting 2 x 2 table.  Then say in one sentence
  how it relates to your Ex 1 table.
	- if Stride = 2, we skip the entire first step, therefore:
		- (0,0) = 4
		- (0,2) = 5
		- (2,0) = 4
		- (2,2) = 4
		- 2 x 2 table:

         4  5  
         4  4  

	- The 2 x 2 output table with Stride 2 consists of the exact values at the 
	four corners of the original 3 x 3 table from EX 1, 
	because sliding by 2 skips the middle row and column.
